In [1]:
import enabol
import hls4ml
print(enabol.__file__)
print(hls4ml.__file__)

[INFO] - ENABOL imported successfully! Version: 0.1.0, URL: https://manuelblancovalentin.github.io/ENABOL/
/Users/mbvalentin/scripts/ENABOL/enabol/__init__.py
/Users/mbvalentin/scripts/ENABOL/hls4ml-trainable/hls4ml/__init__.py


## Global config

In [2]:
## Dataset and model config
N = 1000      # num samples
SEED = 42     # For reproducibility

## Controller config
CONTROLLER_NAME = "none"    # "none", "throttle", "kappa"
CHI = 1.0                   # Only for throttle, ignored for none

## Create the dataset

In [3]:
dataset = enabol.AffineDataset(num_samples=N, use_bias=False, seed=SEED)
X, Y = dataset.get()
print(dataset)

AffineDataset(
 [Input] X: 
    Shape: (1000, 4)
    Dtype: DataType.D2_FLOAT
    X <- Uniform(-1, 1)
 [Output] Y: 
    Shape: (1000, 2)
    Dtype: DataType.D2_FLOAT
    Y <- X @ A.T + b
 ---------
  A = [[ 1.25 -0.75  0.5   0.2 ]
       [-0.4   0.9   1.1  -0.6 ]]
  b = [0. 0.]
----------
Analytic Hessian:
  Lambda max: 1.0822
  Eta max: 1.8481
)


## Create model

In [4]:
model = enabol.LinearBlockModel(
    dataset=dataset,
    num_hidden=[dataset.A.shape[0]],
    activation=None,
    use_batchnorm=False,
    use_bias=False,
    seed=SEED,
)

model.summary()

Model: "LinearBlockModel"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ model_input (InputLayer)        │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense0 (Dense)                  │ (None, 2)              │             8 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8 (32.00 B)

 Trainable params: 8 (32.00 B)

 Non-trainable params: 0 (0.00 B)

## Controller 

In [5]:
# We can start with None (default, no throttling or kappa)
controller = enabol.Controller.from_str(CONTROLLER_NAME, chi=CHI)
print(controller)

CTRL-NONE(
  α = 1
  θ̇ = θ̇_raw
  χ=1 | ρ=0.05 | ε=1e-12 | α∈[0, 1]
)


## Create PredictionDict (for now manually)

In [6]:
from enabol import dtypes, PrecisionDict

def make_precision_dict(BASE_WL, BASE_IWL, QMODE="AP_RND", OMODE="AP_SAT"):
    wide_weight = dtypes.ap_fixed(WL=BASE_WL, IWL=BASE_IWL, QMODE=QMODE, OMODE=OMODE)
    wide_activation = dtypes.ap_fixed(WL=BASE_WL, IWL=BASE_IWL+2, QMODE=QMODE, OMODE=OMODE)
    wide_gradient = dtypes.ap_fixed(WL=BASE_WL+4, IWL=BASE_IWL+2, QMODE=QMODE, OMODE=OMODE)
    wide_update = dtypes.ap_fixed(WL=BASE_WL+4, IWL=BASE_IWL, QMODE=QMODE, OMODE=OMODE)
    wide_accumulator = dtypes.ap_fixed(WL=BASE_WL+12, IWL=BASE_IWL+8, QMODE=QMODE, OMODE=OMODE)
    wide_loss = dtypes.ap_fixed(WL=2*BASE_WL, IWL=BASE_IWL+10, QMODE=QMODE, OMODE=OMODE)
    return PrecisionDict({
                "input": {"value": wide_activation},
                "dense0": {
                    "weight": wide_weight,
                    "activation": wide_activation,
                    "gradient": wide_gradient,
                    "update": wide_update,
                    "accumulator": wide_accumulator,
                },
                "loss": {"value": wide_loss},
            })

precision_dict = make_precision_dict(BASE_WL=16, BASE_IWL=6)
print(precision_dict)

PrecisionDict(
  input:
    value: ap_fixed<16,8,AP_RND,AP_SAT>
  dense0:
    weight: ap_fixed<16,6,AP_RND,AP_SAT>
    activation: ap_fixed<16,8,AP_RND,AP_SAT>
    gradient: ap_fixed<20,8,AP_RND,AP_SAT>
    update: ap_fixed<20,6,AP_RND,AP_SAT>
    accumulator: ap_fixed<28,14,AP_RND,AP_SAT>
  loss:
    value: ap_fixed<32,16,AP_RND,AP_SAT>
)


## Compile the model using enabol/hls4ml

In [7]:
from enabol.compile import compile

hls_model, hls_config = compile(
    model=model,
    dataset=dataset,
    precision=precision_dict,
    backend="Vitis",
    toolchain="auto",          # resolves to kona-vitis-2024_1 on server
    part="xcku035-fbva676-2-e",
    io_type="io_parallel",
    strategy="Latency",
    reuse_factor=1,
    trainable=True,
    optimizer="sgd",
    learning_rate=0.01,
    batch_size=1,
    controller="none",
    write=True,                # generate hls4ml project
    compile_cpp=False,         # local C++ shared-lib compile; no Vitis needed
    build=False,               # actual HLS build/csim/synth; Vitis needed
    csim=True,
    synth=False,
    output_dir = f"../../sandbox/{model.name}_hls",
    overwrite=True,
)

[INFO] - Converting model to hls4ml backend=Vitis, output_dir=../../sandbox/LinearBlockModel_hls
[INFO] - Writing hls4ml project.


In [8]:
print(hls_model.trainable_forward_path)
print(hls_model.trainable_forward_order)
print(hls_model.trainable_backward_order)
print(hls_model.trainable_output_layer)

('model_input', 'dense0')
('dense0',)
('dense0',)
dense0


In [9]:
hls_model.build(csim=True, synth=False)

/bin/sh: vitis-run: command not found


Exception: Build failed for LinearBlockModel. See logs for details.

In [ ]:
from enabol import TestbenchData

tb = TestbenchData.from_dir(hls_model.config.get_output_dir())
tb.plot_training(window_size=30)